# Protein–Ligand MD in 3 commands — `biobb_md_workflows` tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NBDsoftware/biobb_md_workflows/blob/master/notebooks/biobb_md_workflows_colab_tutorial.ipynb)

This notebook sets up and runs a complete **protein–ligand molecular dynamics** simulation
using the high-level **workflows** shipped in
[`biobb_md_workflows`](https://github.com/NBDsoftware/biobb_md_workflows) — *not* the individual
BioExcel building blocks.

The system is the **T4 lysozyme L99A/M102Q** mutant (PDB [`3HTB`](https://www.rcsb.org/structure/3HTB))
in complex with **2-propylphenol** (ligand [`JZ4`](https://www.rcsb.org/ligand/JZ4)) — the same
classic example as the [official GROMACS complex tutorial](http://www.mdtutorials.com/gmx/complex/index.html).

**The whole pipeline is just three workflow commands:**

| Step | Command | What it does |
|------|---------|--------------|
| 1 | `protein_preparation` | Clean, fix and protonate the protein |
| 2 | `ligand_parameterization` | Build GROMACS topology + coordinates for JZ4 |
| 3 | `md_gromacs` | Setup → energy minimization → NVT/NPT equilibration → production MD → analysis |

Each of these replaces ~10–15 manual building-block calls. We then **visualize the trajectory**
and **plot the analysis** the MD workflow produces automatically.

> New to the package? Every command supports `--help`, and full docs live at
> <https://nbdsoftware.github.io/biobb_md_workflows/>.

## 1. Set up the environment

**Running on Google Colab?** The cell below installs everything: it downloads Miniforge, builds
the `biobb_md` conda environment (GROMACS, AmberTools, acpype, the BioBB packages and the
workflows themselves) and installs the visualization libraries into the notebook kernel.

⏳ **This takes ~10–20 minutes** — the conda solve is the slow part. Run it once and wait.

If you are running this **locally** instead, just activate an environment created from the repo's
`environment.yml` (plus GROMACS/AmberTools/acpype available on your `PATH`) and skip this cell.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    # Clone the repo (for the input data and the environment file)
    !git clone https://github.com/NBDsoftware/biobb_md_workflows.git

    # Install Miniforge + mamba
    !wget -q https://github.com/conda-forge/miniforge/releases/download/26.1.0-0/Miniforge3-26.1.0-0-Linux-x86_64.sh
    !bash Miniforge3-26.1.0-0-Linux-x86_64.sh -b -p /usr/local/miniforge

    # Build the biobb_md environment (GROMACS, AmberTools, acpype, BioBB + workflows)
    !/usr/local/miniforge/condabin/mamba env create -f biobb_md_workflows/notebooks/colab_environment.yml

    # Put the env's binaries first on PATH so the workflow CLIs (and gmx, antechamber, ...) resolve
    os.environ['PATH'] = '/usr/local/miniforge/envs/biobb_md/bin:' + os.environ['PATH']

    # Visualization libraries go into the *Colab* kernel (its own Python), not the conda env
    !pip install -q nglview MDAnalysis
    from google.colab import output
    output.enable_custom_widget_manager()

    # Work from inside the cloned repo so that data/3HTB.pdb is available
    os.chdir('biobb_md_workflows')

print('Working directory:', os.getcwd())
print('protein_preparation on PATH:', bool(__import__('shutil').which('protein_preparation')))

## 2. Inputs

We use the PDB complex bundled in the repo at `data/3HTB.pdb` (it already contains both the
protein and the JZ4 ligand). JZ4 (2-propylphenol) is **neutral**, so its formal charge is `0`.

In [ ]:
pdb_code    = "3HTB"
ligand_code = "JZ4"
ligand_charge = 0
input_pdb   = "data/3HTB.pdb"

assert os.path.exists(input_pdb), f"{input_pdb} not found - is the working dir the repo root?"
print("Input structure:", input_pdb)

Let's have a first look at the raw crystal structure (protein + ligand + crystallographic waters):

In [ ]:
import nglview
view = nglview.show_structure_file(input_pdb)
view

## 3. Step 1 — Protein preparation

The `protein_preparation` workflow takes the raw PDB and returns a clean, MD-ready protein:
it extracts the protein, fixes structural defects (alternate locations, missing side chains,
disulfide bonds, amides, chirality), and assigns **protonation states** of titratable residues
at the requested pH (via propka + reduce).

We ask for **GROMACS-style** residue naming (`--output_format gromacs`) because the MD step uses
GROMACS. No Modeller license key is needed here: 3HTB has no missing backbone, so Modeller is
never invoked.

The heteroatoms (the ligand) are stripped from the protein output — that's expected. The ligand
is parameterized separately in the next step, directly from the original PDB.

In [ ]:
!protein_preparation \
    --input_pdb data/3HTB.pdb \
    --output prot_prep \
    --ph 7 \
    --output_format gromacs

The final prepared protein is written to the top of the output folder, named after the input
(`prot_prep/3HTB.pdb`). Intermediate per-step folders (`step1_extractAtoms/`, `step13_propka/`, …)
and the auto-generated `config.yml` are kept alongside for inspection.

In [ ]:
prepared_protein = "prot_prep/3HTB.pdb"
assert os.path.exists(prepared_protein)
print("Prepared protein:", prepared_protein)
nglview.show_structure_file(prepared_protein)

## 4. Step 2 — Ligand parameterization

`ligand_parameterization` generates a **GROMACS topology + coordinates** for the ligand. With no
custom AMBER parameters supplied, it takes the GAFF route: protonate → energy-minimize →
`acpype` (antechamber/GAFF).

We point it at the **original** `data/3HTB.pdb` (which still contains the JZ4 heteroatoms),
restrict it to `JZ4`, give its charge, and request GROMACS format.

In [ ]:
!ligand_parameterization \
    --input_pdb data/3HTB.pdb \
    --output lig_param \
    --ligands JZ4 \
    --charges JZ4:0 \
    --format gromacs

The deliverable is the `topologies/` folder, with one `.itp` + `.gro` pair per ligand:

In [ ]:
ligands_folder = "lig_param/topologies"
print(os.listdir(ligands_folder))
assert os.path.exists(os.path.join(ligands_folder, "JZ4.itp"))
assert os.path.exists(os.path.join(ligands_folder, "JZ4.gro"))

## 5. Step 3 — MD simulation

`md_gromacs` runs the full GROMACS pipeline in four sections:

1. **`1_setup`** — build topology (`pdb2gmx`), insert the ligand + restraints, solvate, add ions
2. **`2_equil`** — energy minimization, NVT and NPT equilibration
3. **`3_prod`** — production MD
4. **`4_analysis`** — RMSD / radius of gyration / RMSF, then trajectory post-processing
   (strip solvent, center, image, fit)

The outputs of the two previous steps chain straight in: the prepared protein as `--input_pdb`
and the ligand topology folder as `--ligands_folder`.

We use **very short** equilibration/production times (`0.1 ns` each) so the demo finishes quickly.
GROMACS from conda is **CPU-only** here, so even this short run may take several minutes. For real
science you would raise `--prod_time` (and run on GPU).

In [ ]:
!md_gromacs \
    --input_pdb prot_prep/3HTB.pdb \
    --ligands_folder lig_param/topologies \
    --output md \
    --gmx_bin gmx \
    --equil_time 0.1 \
    --prod_time 0.1

## 6. Visualize the trajectory

The MD workflow already post-processed the raw trajectory for us: it removed the solvent, and
centered/imaged/fitted the solute. We load the **dry reference structure** together with the
**fitted trajectory** and view it with NGLView (via MDAnalysis, which works in Colab).

In [ ]:
import MDAnalysis as mda
import nglview

topology   = "md/4_analysis/step6_dry_str/dry_structure.pdb"
trajectory = "md/4_analysis/step10_fit_traj/fitted_traj.xtc"

u = mda.Universe(topology, trajectory)
print(f"{len(u.trajectory)} frames, {len(u.atoms)} atoms")

view = nglview.show_mdanalysis(u)
view.add_representation("licorice", selection="resname JZ4")  # highlight the ligand
view

## 7. Analysis plots

`md_gromacs` writes standard analysis curves as `.xvg`/`.xmgr` files under `4_analysis/`. Here we
read them and plot with Plotly:

* **RMSD** of the protein backbone vs. the equilibrated and experimental structures
* **Radius of gyration** (compactness) over time
* **RMSF** (per-residue flexibility)

In [ ]:
import numpy as np
import plotly.graph_objs as go
from plotly.subplots import make_subplots

def read_xvg(path):
    """Load an .xvg/.xmgr file, skipping GROMACS comment/legend lines."""
    data = np.loadtxt(path, comments=['#', '@'])
    return data[:, 0], data[:, 1]

A = "md/4_analysis"
rmsd_eq_x,  rmsd_eq_y  = read_xvg(f"{A}/step2_rmsd_equilibrated/md_rmsdfirst.xvg")
rmsd_exp_x, rmsd_exp_y = read_xvg(f"{A}/step3_rmsd_experimental/md_rmsdexp.xvg")
rgyr_x,     rgyr_y     = read_xvg(f"{A}/step4_rgyr/md_rgyr.xvg")
rmsf_x,     rmsf_y     = read_xvg(f"{A}/step5_rmsf/md_rmsf.xmgr")

fig = make_subplots(rows=2, cols=2, subplot_titles=(
    "Backbone RMSD (vs equilibrated)", "Backbone RMSD (vs experimental)",
    "Radius of gyration", "RMSF (per residue)"))
fig.add_trace(go.Scatter(x=rmsd_eq_x,  y=rmsd_eq_y),  row=1, col=1)
fig.add_trace(go.Scatter(x=rmsd_exp_x, y=rmsd_exp_y), row=1, col=2)
fig.add_trace(go.Scatter(x=rgyr_x,     y=rgyr_y),     row=2, col=1)
fig.add_trace(go.Scatter(x=rmsf_x,     y=rmsf_y),     row=2, col=2)
fig.update_xaxes(title_text="time (ps)", row=1, col=1)
fig.update_xaxes(title_text="time (ps)", row=1, col=2)
fig.update_xaxes(title_text="time (ps)", row=2, col=1)
fig.update_xaxes(title_text="residue",   row=2, col=2)
fig.update_yaxes(title_text="nm", row=1, col=1); fig.update_yaxes(title_text="nm", row=1, col=2)
fig.update_yaxes(title_text="nm", row=2, col=1); fig.update_yaxes(title_text="nm", row=2, col=2)
fig.update_layout(height=650, showlegend=False, title_text="MD analysis — 3HTB / JZ4")
fig.show()

## 8. Wrap-up

In three workflow commands you went from a raw PDB complex to an analyzed MD trajectory:

```
protein_preparation  →  ligand_parameterization  →  md_gromacs
```

What replaced dozens of manual building-block calls is now three self-documenting CLIs. To go
further:

* Run any command with `--help` to see all options (force field, ion concentration, temperature,
  GPU offload with `--use_gpu`, multi-node MPI, PLUMED, …).
* `traj_postprocessing` — the fourth workflow — post-processes a trajectory produced elsewhere
  (strip solvent, center, image, fit), the same cleanup `md_gromacs` runs internally.
* Full documentation: <https://nbdsoftware.github.io/biobb_md_workflows/>.